# `dbnomics(series, start, end=None)` and `dbnomics_series(dataset=None, frequency=None)`

These functions give international macro series from `indicators.dbnomics`, oldest first. DBnomics collects the data from its original providers: the OECD, Eurostat, the BIS, the European Commission (AMECO), the World Bank and the ECB. The series are leading indicators, confidence, inflation, growth, rates, credit and house prices, mostly for the euro area and the G7 countries.

`dbnomics_series()` gives the dataset, name, units, frequency, category, country and stale flag of each series id. Each cell reads prod as `readonly_user`, and does not write. You can change the arguments and run a cell again.

Use the DBnomics series code (`M.U2.N.000000.4D0.ANR`) or the name of a series, in any case and spacing. The functions do not give periods with no value, and do not fill them.

> **About the data shown.** Retrieved via DBnomics (https://db.nomics.world, accessed November 27, 2026). DBnomics' own datasets are under the Open Database License (ODbL). Each series keeps the licence of its original provider: OECD (CC BY 4.0), Eurostat (free reuse with the source acknowledged), BIS (free use with BIS cited; a commercial product must not charge its users more because the BIS data is in it), European Commission AMECO (free reuse, Commission Decision 2011/833/EU), World Bank (CC BY 4.0) and ECB (free reuse with the source quoted). Sources: OECD, Eurostat, BIS, European Commission, World Bank and ECB, via DBnomics. arora_data has no affiliation with DBnomics or with these providers, and they do not endorse it. This notebook is not investment advice.

In [1]:
from datetime import date, timedelta

import polars as pl

from arora_data.dates import today_et
from arora_data.dbnomics import DbnomicsSeries, DbnomicsValue
from arora_data.frames import to_df
from arora_data.sync import dbnomics, dbnomics_series

## The series
Each call reads the series from the table, thus a series that the main repo adds or removes needs no code change. The labels come from the latest row of each series. `first_date` and `latest_date` are the first and latest periods that have a value. `country` is the area code in the series code, as the provider writes it (`U2`, `XM`, `EA20` and `I10` are codes for the euro area).

In [2]:
series = to_df(dbnomics_series(), DbnomicsSeries)
print(series.height, "series")
series.select("dataset", "series_id", "country", "frequency", "latest_date", "stale")

62 series


dataset,series_id,country,frequency,latest_date,stale
str,str,str,str,date,bool
"""WB/WDI""","""A-FP.CPI.TOTL.ZG-WLD""","""WLD""","""annual""",2023-01-01,true
"""WB/WDI""","""A-NY.GDP.MKTP.KD.ZG-CHN""","""CHN""","""annual""",2023-01-01,true
"""WB/WDI""","""A-NY.GDP.MKTP.KD.ZG-IND""","""IND""","""annual""",2023-01-01,true
"""WB/WDI""","""A-NY.GDP.MKTP.KD.ZG-WLD""","""WLD""","""annual""",2023-01-01,true
"""WB/WDI""","""A-NY.GDP.PCAP.CD-CHN""","""CHN""","""annual""",2023-01-01,true
…,…,…,…,…,…
"""BIS/WS_SPP""","""Q.XM.R.628""","""XM""","""quarterly""",2024-10-01,true
"""ECB/MNA""","""Q.Y.I10.W2.S1.S1.B.B1GQ._Z._Z.…","""I10""","""quarterly""",2026-07-01,false
"""OECD/DSD_STES@DF_CLI""","""USA.M.BCICP.IX._Z.AA.IX._Z.H""","""USA""","""monthly""",2026-04-01,true


## Stale series
`stale` is True when today (US/Eastern) is after the latest date with a value plus 2 periods of the frequency plus 120 days. DBnomics stopped its reads of some providers, thus many series stop. The number of stale series for each provider, and their latest dates:

In [3]:
series.with_columns(pl.col("dataset").str.split("/").list.first().alias("provider")) \
    .group_by("provider").agg(pl.len().alias("series"), pl.col("stale").sum().alias("stale"),
                              pl.col("latest_date").min().alias("oldest latest"), pl.col("latest_date").max().alias("newest latest")) \
    .sort("provider")

provider,series,stale,oldest latest,newest latest
str,u32,u32,date,date
"""AMECO""",2,0,2028-01-01,2028-01-01
"""BIS""",6,6,2024-10-01,2025-06-01
"""ECB""",7,0,2026-04-01,2026-10-31
"""Eurostat""",5,5,2025-01-01,2025-12-01
"""OECD""",36,36,2026-03-01,2026-05-01
"""WB""",6,6,2023-01-01,2023-01-01


In [4]:
for s in dbnomics_series():
    if not s.stale:
        print(f"{s.latest_date}  {s.dataset:<14} {s.series_id}")

2026-10-31  ECB/FM         D.U2.EUR.4F.KR.DFR.LEV
2028-01-01  AMECO/AVGDGP   EA20.1.0.0.0.AVGDGP
2028-01-01  AMECO/ZUTN     EA20.1.0.0.0.ZUTN
2026-09-01  ECB/EXR        M.E03.EUR.ERC0.A
2026-08-01  ECB/STBS       M.I10.Y.PROD.NS0020.4D0.N.IX
2026-09-01  ECB/HICP       M.U2.N.000000.4D0.ANR
2026-04-01  ECB/RESR       Q.U2._T.N._TR.TVAL.4D0.TB.N.IX
2026-04-01  ECB/RDF        Q.U2.EUR.4F.CR.DCGDPG.RO
2026-07-01  ECB/MNA        Q.Y.I10.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.G1


## One dataset or one frequency
`dbnomics_series(dataset=...)` and `dbnomics_series(frequency=...)` give only the series that have that value. You can use the two filters together. The number of series for each dataset and frequency:

In [5]:
series.group_by("dataset", "frequency").agg(pl.len().alias("series")).sort("dataset")

dataset,frequency,series
str,str,u32
"""AMECO/AVGDGP""","""annual""",1
"""AMECO/ZUTN""","""annual""",1
"""BIS/WS_CBPOL""","""monthly""",1
"""BIS/WS_CREDIT_GAP""","""quarterly""",1
"""BIS/WS_DSR""","""quarterly""",1
…,…,…
"""Eurostat/namq_10_gdp""","""quarterly""",1
"""Eurostat/prc_hicp_manr""","""monthly""",1
"""Eurostat/une_rt_m""","""monthly""",1


In [6]:
for s in dbnomics_series(dataset="OECD/DSD_STES@DF_CLI", frequency="monthly")[:6]:
    print(f"{s.country:<5} {s.series_id:<32} {s.category}")

BRA   BRA.M.BCICP.IX._Z.AA.IX._Z.H     Sentiment
BRA   BRA.M.CCICP.IX._Z.AA.IX._Z.H     Sentiment
BRA   BRA.M.LI.IX._Z.AA.IX._Z.H        Leading Indicators
CHN   CHN.M.BCICP.IX._Z.AA.IX._Z.H     Sentiment
CHN   CHN.M.CCICP.IX._Z.AA.IX._Z.H     Sentiment
CHN   CHN.M.LI.IX._Z.AA.IX._Z.H        Leading Indicators


## One series, by id or by name
Euro-area HICP inflation (ECB), the annual rate of change in percent, for the last 12 months. The id and the name, in any case and spacing, give the same rows. A monthly value has the first day of its month.

In [7]:
hicp = dbnomics_series(dataset="ECB/HICP")[0]
year = today_et() - timedelta(days=366)
assert dbnomics("M.U2.N.000000.4D0.ANR", year) == dbnomics("m.u2.n.000000.4d0.anr", year) == dbnomics(hicp.name.upper(), year)
to_df(dbnomics("M.U2.N.000000.4D0.ANR", year), DbnomicsValue).select("date", "value", "units")

date,value,units
date,"decimal[38,8]",str
2025-12-01,2.00000000,"""percent"""
2026-01-01,1.70000000,"""percent"""
2026-02-01,1.90000000,"""percent"""
2026-03-01,2.60000000,"""percent"""
2026-04-01,3.00000000,"""percent"""
2026-05-01,3.20000000,"""percent"""
2026-06-01,2.80000000,"""percent"""
2026-07-01,2.90000000,"""percent"""
2026-08-01,3.20000000,"""percent"""


## The ECB deposit facility rate
A daily series. The days where the rate changed since 2022:

In [8]:
dfr = to_df(dbnomics("D.U2.EUR.4F.KR.DFR.LEV", date(2022, 1, 1)), DbnomicsValue).select("date", "value")
dfr.filter(pl.col("value") != pl.col("value").shift(1))

date,value
date,"decimal[38,8]"
2022-07-27,0.00000000
2022-09-14,0.75000000
2022-11-02,1.50000000
2022-12-21,2.00000000
2023-02-08,2.50000000
…,…
2025-03-12,2.50000000
2025-04-23,2.25000000
2025-06-11,2.00000000


## OECD composite leading indicators
The latest value of the composite leading indicator (CLI) for each country or group, with its date. These series are stale: DBnomics has no OECD data after the date shown.

In [9]:
cli = [s for s in dbnomics_series(dataset="OECD/DSD_STES@DF_CLI") if ".M.LI." in s.series_id]
pl.DataFrame([{"country": s.country, "date": s.latest_date, "CLI": dbnomics(s.series_id, s.latest_date)[-1].value, "stale": s.stale}
              for s in cli]).sort("country")

country,date,CLI,stale
str,date,"decimal[38,8]",bool
"""BRA""",2026-05-01,103.82030000,true
"""CHN""",2026-05-01,98.90580000,true
"""DEU""",2026-05-01,100.83810000,true
"""G4E""",2026-05-01,100.64760000,true
"""G7""",2026-05-01,100.84550000,true
"""GBR""",2026-05-01,100.50010000,true
"""IND""",2026-05-01,102.10460000,true
"""JPN""",2026-05-01,100.27400000,true
"""KOR""",2026-05-01,102.60060000,true


## AMECO forecasts
The AMECO values from the current year on are forecasts of the European Commission, not outturns. Each AMECO release writes over them. The forecasts go past today, thus the call gives an end after today. The euro-area unemployment rate, in percent of the active population:

In [10]:
to_df(dbnomics("EA20.1.0.0.0.ZUTN", date(2020, 1, 1), date(2030, 12, 31)), DbnomicsValue).select("date", "value") \
    .with_columns((pl.col("date").dt.year() >= today_et().year).alias("forecast"))

date,value,forecast
date,"decimal[38,8]",bool
2020-01-01,8.00000000,false
2021-01-01,7.80000000,false
2022-01-01,6.80000000,false
2023-01-01,6.60000000,false
2024-01-01,6.40000000,false
2025-01-01,6.40000000,false
2026-01-01,6.50000000,true
2027-01-01,6.60000000,true
2028-01-01,6.60000000,true


## Units
The units of the series:

In [11]:
series.group_by("units").agg(pl.len().alias("series")).sort("units")

units,series
str,u32
"""USD""",2
"""index""",43
"""percent""",13
"""percent of GDP""",1
"""percent of potential GDP""",1
"""percentage points""",2


## Revisions
The main repo reads the full history of each series on each run. Thus the revisions (for example the monthly OECD CLI revisions) replace the earlier values, and the table does not keep the earlier values.

## Not here
`fred()`, `bls()` and `bea()` give the US data. The table does not hold UK, Japan or Australia data from their national offices now.

## Bad input
An unknown id or name, a filter value that no series has, or a start after the end, raises a clear error before the function reads values.

In [12]:
for call in [lambda: dbnomics("HICP", date(2026, 1, 1)), lambda: dbnomics("M.U2", date(2026, 1, 1)),
             lambda: dbnomics("M.XM", date(2026, 1, 2), date(2026, 1, 1)),
             lambda: dbnomics_series(frequency="weekly"), lambda: dbnomics_series(dataset="OECD")]:
    try:
        call()
    except ValueError as e:
        print(e)

'HICP' is not a stored DBnomics series: use an id or name from dbnomics_series(), such as USA.M.LI.IX._Z.AA.IX._Z.H


'M.U2' is not a stored DBnomics series: use an id or name from dbnomics_series(), such as USA.M.LI.IX._Z.AA.IX._Z.H
start must be on or before end


'weekly' is not a stored DBnomics frequency: use one of annual, daily, monthly, quarterly


'OECD' is not a stored DBnomics dataset: use one of AMECO/AVGDGP, AMECO/ZUTN, BIS/WS_CBPOL, BIS/WS_CREDIT_GAP, BIS/WS_DSR, BIS/WS_EER, BIS/WS_SPP, BIS/WS_TC, ECB/EXR, ECB/FM, ECB/HICP, ECB/MNA, ECB/RDF, ECB/RESR, ECB/STBS, Eurostat/STS_INPR_M, Eurostat/ei_bssi_m_r2, Eurostat/namq_10_gdp, Eurostat/prc_hicp_manr, Eurostat/une_rt_m, OECD/DSD_STES@DF_CLI, WB/WDI
